# MosekChordalSDP

`MosekChordalSDP` uses symbolic elimination to replace one large positive-semidefinite cone with overlapping clique cones, while exposing the same solve and QCQP recovery lifecycle as the monolithic formulation. After `solve()`, `qcqpValues()` and `variableEVRs()` are optional, independent functional queries for recovered vectors and rank-one diagnostics.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415  
All Rights Reserved  
Authors: Frank Dellaert, et al. (see THANKS for the full author list)  
See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/certifiable/doc/MosekChordalSDP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
from gtsam.symbol_shorthand import X

## Chordal decomposition

The aggregate sparsity graph of the lifted QCQP is completed according to either `ChordalOrderingType.Metis` or `ChordalOrderingType.Colamd`. Symbolic elimination produces a Bayes tree; each clique defines a smaller PSD cone, and separator consistency constraints make the clique matrices agree on overlaps.

METIS is generally the scalable default. COLAMD is useful for deterministic comparisons and small problems. `bayesTree()` exposes the symbolic decomposition chosen by the solver.

In [3]:
num_poses = 3
step = gtsam.Pose3(
    gtsam.Rot3.Rz(2.0 * np.pi / num_poses), np.array([2.0, 0.0, 0.0])
)
ground_truth = [gtsam.Pose3()]
for _ in range(1, num_poses):
    ground_truth.append(ground_truth[-1].compose(step))

graph = gtsam.NonlinearFactorGraph()
graph.add(gtsam.FrobeniusPriorPose3(
    X(0), ground_truth[0].matrix(), gtsam.noiseModel.Constrained.All(16)
))
for i in range(num_poses):
    j = (i + 1) % num_poses
    graph.add(gtsam.FrobeniusBetweenFactorPose3(
        X(i), X(j), ground_truth[i].between(ground_truth[j])
    ))
problem = gtsam.QcqpProblem(graph)

In [4]:
if not hasattr(gtsam, "MosekChordalSDP"):
    print("This GTSAM build does not include the optional MOSEK backend.")
else:
    solver = gtsam.MosekChordalSDP(problem, gtsam.ChordalOrderingType.Metis)
    if not solver.solve({"intpntCoTolRelGap": 1e-10}):
        raise RuntimeError("MOSEK did not return a readable primal solution")
    recovered = gtsam.extractQcqpValuesPose3(solver.qcqpValues())
    errors = [
        np.linalg.norm(ground_truth[i].localCoordinates(recovered.atPose3(X(i))))
        for i in range(num_poses)
    ]
    print("status:", solver.problemStatus())
    print("objective:", solver.objectiveValue())
    print("solve time (s):", solver.solveTimeSeconds())
    print("Bayes-tree cliques:", solver.bayesTree().size())
    print("ordered dimensions:", solver.orderedKeyDims())
    print("eigenvalue ratios:", solver.variableEVRs())
    print("maximum Pose3 error:", max(errors))

status: ProblemStatus::PrimalAndDualFeasible
objective: 2.1758594925813668e-11
solve time (s): 0.018308162689208984
Bayes-tree cliques: 1
ordered dimensions: {8646911284551352320: 13, 8646911284551352321: 13, 8646911284551352322: 13}
eigenvalue ratios: [12302689276843.451, 2602396961361.1143, 1147993372952.9626]
maximum Pose3 error: 4.1463411520090023e-07


Use the monolithic solver as a simple reference and the chordal solver when sparsity makes the clique cones materially smaller than the full lifted matrix. Both recover the same D=1 QCQP representation and therefore use the same typed extraction functions.